# Baseline Controllers: On/Off vs PID (BASE-01..03)

Closed-loop 24h runs of the hysteresis on/off and dual-loop PID baselines (`src/onoff_controller.py`, `src/pid_controller.py`) against the Phase-1 plant, tracking fixed setpoints 25 C / 70 %RH on the seed-7 weather. Both controllers return the shared `{heater, fan_cooler, fogger, vent}` duty dict fed directly to `step()`; each run owns a separate controller state (never shared).

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.greenhouse_model import step, load_weather_profile, daytime_solar, params, relative_humidity, wsat_g_per_kg
from src.weather_profile_generator import generate_weather_profile
from src.onoff_controller import onoff_step
from src.pid_controller import pid_step

T_SP, RH_SP = 25.0, 70.0
print("params:", params)

In [ ]:
# Weather input: regenerate the seed-7 profile only if absent, then load.
import os
CSV = "weather/weather_profile.csv"
if not os.path.exists(CSV):
    generate_weather_profile(CSV, seed=7)

t_min, T_out, w_out = load_weather_profile(CSV)
hours = np.arange(0, 1440) / 60.0          # dt = 60 s -> one step per minute
T_out_hr = np.interp(hours, t_min, T_out)
w_out_hr = np.interp(hours, t_min, w_out)

In [ ]:
# Closed loop, once per baseline, over the identical disturbance.
# Separate controller states per run; setpoints pass as function args.
def run_closed_loop(controller_step):
    T, w = 20.0, 0.65 * wsat_g_per_kg(20.0)
    st = None
    T_hist, RH_hist, U_hist = [], [], []
    for i, h in enumerate(hours):
        rh = relative_humidity(w, T)
        act, st = controller_step(T, rh, T_SP, RH_SP, st)
        ws = (T_out_hr[i], w_out_hr[i], daytime_solar(h, params["A_sol"]))
        T, w, rh = step(60.0, act, ws, T, w)
        T_hist.append(T); RH_hist.append(rh); U_hist.append(act)
    return T_hist, RH_hist, U_hist

T_on, RH_on, U_on = run_closed_loop(onoff_step)
T_pid, RH_pid, U_pid = run_closed_loop(pid_step)

for name, T_hist, RH_hist in (("onoff", T_on, RH_on), ("pid", T_pid, RH_pid)):
    iae = sum(abs(T - T_SP) for T in T_hist) / 60.0
    print("%s: T [%.2f, %.2f]  RH [%.1f, %.1f]  T-IAE %.2f C-h" %
          (name, min(T_hist), max(T_hist), min(RH_hist), max(RH_hist), iae))

**Honest baseline weakness (D-03):** RH pins at 100 % for long stretches under both baselines. The fogger injects ~0.44 g/kg/min while the only moisture sinks (T-slaved vent exchange, cooler condensation) follow the temperature loop, so overnight saturation is structural — not a tuning defect. Loops stay independent (no vent-on-humid workaround); this is exactly the weakness the fuzzy controller will exploit.

In [ ]:
# Per-controller panels: T vs setpoint, RH vs setpoint, duty stacks.
SP_T = np.full_like(hours, T_SP)
SP_RH = np.full_like(hours, RH_SP)
plt.figure(figsize=(12, 10))
for col, (name, T_hist, RH_hist, U_hist) in enumerate((("on/off", T_on, RH_on, U_on), ("PID", T_pid, RH_pid, U_pid))):
    plt.subplot(3, 2, 1 + col)
    plt.plot(hours, T_hist, label="inside T (%s)" % name)
    if col == 0:
        plt.plot(hours, T_out_hr, label="outdoor T", linestyle="--")
    plt.plot(hours, SP_T, label="setpoint", linestyle=":")
    plt.ylabel("Temperature (degC)"); plt.legend(fontsize=8); plt.grid(alpha=0.3)

    plt.subplot(3, 2, 3 + col)
    plt.plot(hours, RH_hist, color="C2", label="inside RH (%s)" % name)
    plt.plot(hours, SP_RH, label="setpoint", linestyle=":")
    plt.ylabel("RH (%)"); plt.legend(fontsize=8); plt.grid(alpha=0.3)

    plt.subplot(3, 2, 5 + col)
    for key in ("heater", "fan_cooler", "fogger", "vent"):
        plt.plot(hours, [u[key] for u in U_hist], label=key)
    plt.xlabel("hour of day"); plt.ylabel("duty"); plt.legend(fontsize=8); plt.grid(alpha=0.3)

plt.suptitle("Baseline controllers - 24h closed-loop (on/off vs PID, 25 C / 70 %RH)")
plt.tight_layout()
plt.savefig("notebooks/02_baselines.png", dpi=120)
plt.show()